# Phase 4: Court Judgement Prediction

 _This is submission 2 where we fine tune our phase 1 Selective SSM with Prompt tuning adaptation for downstream task of court Judgement Prediction._

## 1. Login and GPU check

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Dataloading and Pre-processing

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone" 
MAX_LENGTH= 1024
HEAD_LENGTH= 256
TAIL_LENGTH= 766
# --- 1. Data Loading ---
print("Loading datasets from Hugging Face Hub...")

try:
    # Explicitly specify 'phase_4' here
    hf_dataset = load_dataset(HF_DATASET_REPO, 'phase_4')
    print(f"Successfully loaded splits: {list(hf_dataset.keys())}")
except Exception as e:
    print(f"Error loading dataset: {e}")

def standardize_labels(example, is_test=False):
    if not is_test:
        if 'pred' in example and 'label' not in example:
             example['label'] = example['pred']
    return example

# Apply standard formatting
for split in hf_dataset.keys():
    is_test_split = (split == 'test')
    hf_dataset[split] = hf_dataset[split].map(lambda x: standardize_labels(x, is_test=is_test_split))

print("Data loading complete.")

Loading datasets from Hugging Face Hub...


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

... (6 outputs left out)

Map:   0%|          | 0/37387 [00:00<?, ? examples/s]

Map:   0%|          | 0/1988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1517 [00:00<?, ? examples/s]

Data loading complete.


In [4]:
# --- 2. Preprocessing & Strategic Truncation ---

from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

BPE_FILE = "/kaggle/input/datasets/vaibhavim081203/phase3feed/cs781_phase1/legal_bpe.json"

# Load backend tokenizer directly
backend_tok = Tokenizer.from_file(BPE_FILE)

# Wrap into Hugging Face PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend_tok)

# Define special tokens according to your Phase 1 setup
tokenizer.pad_token = "[PAD]" if "[PAD]" in tokenizer.get_vocab() else tokenizer.eos_token
tokenizer.cls_token = "[CLS]" if "[CLS]" in tokenizer.get_vocab() else "<s>"
tokenizer.sep_token = "[SEP]" if "[SEP]" in tokenizer.get_vocab() else "</s>"

print(f"Tokenizer loaded successfully! Vocab size: {len(tokenizer)}")

# Safe fallback for special tokens
START_ID = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else (tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0)
END_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

def strategic_truncate_tokenize(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        padding=False,
        truncation=False,
        add_special_tokens=False
    )
    
    input_ids_list = []
    attention_mask_list = []
    
    for input_ids in tokenized_inputs["input_ids"]:
        if len(input_ids) > (MAX_LENGTH - 2):
            head_tokens = input_ids[:HEAD_LENGTH]
            tail_tokens = input_ids[-TAIL_LENGTH:]
            truncated_ids = head_tokens + tail_tokens
        else:
            truncated_ids = input_ids
            
        # Add Special Tokens safely without injecting None
        final_ids = [START_ID] + truncated_ids + [END_ID]
        
        attention_mask = [1] * len(final_ids)
        
        input_ids_list.append(final_ids)
        attention_mask_list.append(attention_mask)
        
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list
    }

print("Re-applying strategic truncation...")
tokenized_datasets = hf_dataset.map(
    strategic_truncate_tokenize,
    batched=True,
    remove_columns=["text"] 
)


Tokenizer loaded successfully! Vocab size: 32000
Re-applying strategic truncation...


Map:   0%|          | 0/37387 [00:00<?, ? examples/s]

Map:   0%|          | 0/1988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1517 [00:00<?, ? examples/s]

## 3. Model Initialization and Prompt tuning

In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight


def selective_scan(dt, A, B, C, x, chunk: int = 64):
    """
    h_t = exp(dt_t * A) * h_{t-1} + dt_t * B_t * x_t ;  y_t = sum_n C_t[n] * h_t[:, n]

    Same maths as a step-by-step loop over all L positions, but done in two levels so that
    only `chunk + L/chunk` python-level steps are needed instead of L (2,048 -> ~96):
      1. inside every chunk, run the recurrence from h=0, all chunks in parallel;
      2. a short sequential pass over the chunks produces the state entering each chunk;
      3. add that carried state's contribution, exp(cumsum(dt*A)) * carry, to every position.
    Every exponent is <= 0 (A < 0, dt >= 0), so nothing can overflow.
    All inputs must be float32.  dt, x: (b, L, d)   A: (d, n)   B, C: (b, L, n)  ->  y: (b, L, d)
    """
    b, L, d = x.shape
    n = A.shape[1]
    pad = (-L) % chunk
    if pad:  # dt = 0 -> decay 1 and no input: the state is left untouched by the padded steps
        dt = F.pad(dt, (0, 0, 0, pad))
        x = F.pad(x, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, pad))
    nc = (L + pad) // chunk
    dt = dt.reshape(b, nc, chunk, d)
    x = x.reshape(b, nc, chunk, d)
    B = B.reshape(b, nc, chunk, n)
    C = C.reshape(b, nc, chunk, n)

    dtA = dt.unsqueeze(-1) * A                                   # (b, nc, T, d, n)
    dA = torch.exp(dtA)
    dBx = dt.unsqueeze(-1) * x.unsqueeze(-1) * B.unsqueeze(3)    # (b, nc, T, d, n)

    # 1. local scan inside every chunk (starting from zero state)
    h = x.new_zeros(b, nc, d, n)
    ys = []
    for t in range(chunk):
        h = dA[:, :, t] * h + dBx[:, :, t]
        ys.append((h * C[:, :, t].unsqueeze(2)).sum(-1))         # (b, nc, d)
    y_local = torch.stack(ys, dim=2)                             # (b, nc, T, d)

    # 2. state entering each chunk
    S = torch.cumsum(dtA, dim=2)                                 # log of the decay accumulated inside the chunk
    P_end = torch.exp(S[:, :, -1])                               # (b, nc, d, n)
    carry = x.new_zeros(b, d, n)
    carries = []
    for c in range(nc):
        carries.append(carry)
        carry = P_end[:, c] * carry + h[:, c]
    carry_in = torch.stack(carries, dim=1)                       # (b, nc, d, n)

    # 3. contribution of the carried state to every position of its chunk
    y_corr = (torch.exp(S) * carry_in.unsqueeze(2) * C.unsqueeze(3)).sum(-1)   # (b, nc, T, d)

    y = (y_local + y_corr).reshape(b, nc * chunk, d)
    return y[:, :L]


class SelectiveSSM(nn.Module):
    """
    Selective State-Space Module (Mamba core): input-dependent (Delta, B, C) and a linear recurrence over positions.
    """
    def __init__(self, d_model: int = 768, d_state: int = 16, expand: int = 2,
                 dt_rank: int = None, kernel_size: int = 4, scan_chunk: int = 64):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_model / 16)
        self.scan_chunk = scan_chunk

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=kernel_size,
                                padding=kernel_size - 1, groups=self.d_inner, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.reset_dt_parameters()

    def reset_dt_parameters(self):
        """Mamba's timescale init: dt in [1e-3, 1e-1] (log-uniform), stored through the inverse softplus."""
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt_min, dt_max = 1e-3, 1e-1
        dt = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        dt = dt.clamp(min=1e-4)
        inv_softplus_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_softplus_dt)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape

        xz = self.in_proj(x)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = x_branch.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :seq_len]       # drop the right-hand causal padding
        x_act = F.silu(x_conv.transpose(1, 2))

        x_ssm = self.x_proj(x_act)
        dt_raw, B, C = torch.split(x_ssm, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))

        # The recurrence always runs in float32 (exp/cumsum/products are not safe in fp16 autocast).
        with torch.autocast(device_type=x.device.type, enabled=False):
            A = -torch.exp(self.A_log.float())
            x32 = x_act.float()
            y = selective_scan(dt.float(), A, B.float(), C.float(), x32, chunk=self.scan_chunk)
            y = y + x32 * self.D
            y = y * F.silu(z_branch.float())
        return self.out_proj(y.to(z_branch.dtype))


def segsum(x: torch.Tensor) -> torch.Tensor:
    """(..., T) -> (..., T, T) with out[i, j] = sum(x[j+1..i]) for i >= j and -inf above the diagonal."""
    T = x.size(-1)
    c = torch.cumsum(x, dim=-1)
    out = c.unsqueeze(-1) - c.unsqueeze(-2)
    return out.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device)), float("-inf"))


def ssd_scan(x, dt, A, B, C, chunk: int = 64):
    """
    Chunked state-space-duality scan (Mamba-2): same recurrence as a sequential loop,
        h_t = exp(A * dt_t) * h_{t-1} + dt_t * (x_t outer B_t) ;   y_t = h_t . C_t
    but computed with a handful of batched matrix multiplications instead of L (or L * d_inner * d_state) tiny steps:
      1. inside every chunk: (C B^T) * decay-mask  ->  matmul with x          (the diagonal blocks)
      2. the state each chunk leaves behind                                   (matmul)
      3. a short recurrence over the chunk states                             (tiny)
      4. the carried state's contribution to every position of its chunk      (matmul)
    All exponents are <= 0 (A < 0, dt >= 0), so nothing can overflow. Inputs must be float32.
    x: (b, L, h, p)   dt: (b, L, h)   A: (h,)   B, C: (b, L, n) shared by all heads   ->   y: (b, L, h, p)
    """
    b, L, h, p = x.shape
    n = B.size(-1)
    pad = (-L) % chunk
    if pad:  # dt = 0 -> decay 1 and no input: padded steps leave the state untouched
        x = F.pad(x, (0, 0, 0, 0, 0, pad))
        dt = F.pad(dt, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, pad))
    nc = (L + pad) // chunk

    X = (x * dt.unsqueeze(-1)).reshape(b, nc, chunk, h, p)
    Bc = B.reshape(b, nc, chunk, n)
    Cc = C.reshape(b, nc, chunk, n)
    Ad = (A.view(1, 1, h) * dt).reshape(b, nc, chunk, h).permute(0, 3, 1, 2)          # (b, h, nc, chunk)
    Acs = torch.cumsum(Ad, dim=-1)

    # 1. diagonal blocks
    Lm = torch.exp(segsum(Ad))                                                         # (b, h, nc, l, s)
    G = torch.einsum("bcln,bcsn->bcls", Cc, Bc)
    Y_diag = torch.einsum("bhcls,bcshp->bclhp", G.unsqueeze(1) * Lm, X)

    # 2. state left behind by each chunk
    decay_states = torch.exp(Acs[..., -1:] - Acs)                                      # (b, h, nc, l)
    states = torch.einsum("bcln,bclhp->bchpn", Bc, X * decay_states.permute(0, 2, 3, 1).unsqueeze(-1))

    # 3. recurrence over chunk states (state entering chunk c = decayed sum of all earlier chunk states)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)               # (b, nc+1, h, p, n)
    decay_chunk = torch.exp(segsum(F.pad(Acs[..., -1], (1, 0))))                       # (b, h, nc+1, nc+1)
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]            # (b, nc, h, p, n)

    # 4. carried state -> outputs
    Y_off = torch.einsum("bcln,bchpn->bclhp", Cc, states) * torch.exp(Acs).permute(0, 2, 3, 1).unsqueeze(-1)

    return (Y_diag + Y_off).reshape(b, nc * chunk, h, p)[:, :L]


class SSDMixer(nn.Module):
    """
    Mamba-2 style selective state-space mixer: input-dependent (dt, B, C), a scalar decay A per head, heads of size
    `headdim`, B and C shared by all heads, gated RMS-norm before the output projection.
    Its scan is matmul-based (see ssd_scan), which is what makes it practical to train without custom CUDA kernels.
    """
    def __init__(self, d_model: int = 768, d_state: int = 64, expand: int = 2, headdim: int = 64,
                 kernel_size: int = 4, scan_chunk: int = 64):
        super().__init__()
        self.d_inner = d_model * expand
        assert self.d_inner % headdim == 0, "d_model * expand must be divisible by headdim"
        self.d_state = d_state
        self.headdim = headdim
        self.nheads = self.d_inner // headdim
        self.scan_chunk = scan_chunk
        conv_dim = self.d_inner + 2 * d_state                       # x, B, C go through the causal conv together

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner + 2 * d_state + self.nheads, bias=False)   # z, x, B, C, dt
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=kernel_size, padding=kernel_size - 1,
                                groups=conv_dim, bias=True)
        self.dt_bias = nn.Parameter(torch.zeros(self.nheads))
        self.A_log = nn.Parameter(torch.zeros(self.nheads))
        self.D = nn.Parameter(torch.ones(self.nheads))
        self.norm = RMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.reset_dt_parameters()

    def reset_dt_parameters(self):
        """dt in [1e-3, 1e-1] (log-uniform) through the inverse softplus; A in [1, 16] (log-uniform in magnitude)."""
        dt_min, dt_max = 1e-3, 1e-1
        dt = torch.exp(torch.rand(self.nheads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        with torch.no_grad():
            self.dt_bias.copy_(dt + torch.log(-torch.expm1(-dt)))
            self.A_log.copy_(torch.log(torch.empty(self.nheads).uniform_(1.0, 16.0)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, L, _ = x.shape
        z, xBC, dt = torch.split(self.in_proj(x), [self.d_inner, self.d_inner + 2 * self.d_state, self.nheads], dim=-1)
        xBC = F.silu(self.conv1d(xBC.transpose(1, 2))[:, :, :L].transpose(1, 2))     # drop the right-hand causal padding
        xs, Bm, Cm = torch.split(xBC, [self.d_inner, self.d_state, self.d_state], dim=-1)

        # The scan always runs in float32 (exp / cumsum / products are not safe in fp16 autocast).
        with torch.autocast(device_type=x.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias)
            A = -torch.exp(self.A_log.float())
            xh = xs.float().reshape(b, L, self.nheads, self.headdim)
            y = ssd_scan(xh, dt, A, Bm.float(), Cm.float(), chunk=self.scan_chunk)
            y = (y + xh * self.D.view(1, 1, -1, 1)).reshape(b, L, self.d_inner)
            y = self.norm(y * F.silu(z.float()))
        return self.out_proj(y.to(z.dtype))


class MambaBlock(nn.Module):
    """Residual block with RMS normalisation around either mixer ("mamba2" = SSDMixer, "mamba1" = SelectiveSSM)."""
    def __init__(self, d_model: int, d_state: int = 64, expand: int = 2, scan_chunk: int = 64,
                 variant: str = "mamba2", headdim: int = 64):
        super().__init__()
        self.norm = RMSNorm(d_model)
        if variant == "mamba2":
            self.ssm = SSDMixer(d_model=d_model, d_state=d_state, expand=expand, headdim=headdim, scan_chunk=scan_chunk)
        elif variant == "mamba1":
            self.ssm = SelectiveSSM(d_model=d_model, d_state=d_state, expand=expand, scan_chunk=scan_chunk)
        else:
            raise ValueError(f"unknown SSM variant: {variant}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.ssm(self.norm(x))


class Model3_SSM(nn.Module):
    """Selective state-space causal language model."""
    def __init__(self, vocab_size: int = 32000, d_model: int = 768, n_layers: int = 12,
                 d_state: int = 64, expand: int = 2, scan_chunk: int = 64, grad_ckpt: bool = False,
                 variant: str = "mamba2", headdim: int = 64):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.grad_ckpt = grad_ckpt

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, expand=expand, scan_chunk=scan_chunk,
                       variant=variant, headdim=headdim)
            for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight      # weight tying

        self.apply(self._init_weights)
        # self.apply() above also hits the mamba1 dt_proj Linear (weights -> N(0, 0.02), bias -> 0) and would wipe
        # the timescale init: put it back (the mamba2 mixer keeps dt_bias / A_log as plain parameters, harmless there).
        for m in self.modules():
            if isinstance(m, (SelectiveSSM, SSDMixer)):
                m.reset_dt_parameters()

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, input_ids=None, inputs_embeds=None, targets=None, last_only=False ,return_hidden=False, **kwargs):
        # If raw embeddings are provided (like from our Prompt Tuning wrapper), use them
        if inputs_embeds is not None:
            x = inputs_embeds
        # Otherwise, fall back to standard token ID embedding
        else:
            x = self.embedding(input_ids) # Use your actual embedding layer name here
       
        for layer in self.layers:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                # recompute the block in backward: the (b, L, d_inner, d_state) scan tensors are never kept for all layers
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
                
        x = self.norm_f(x)
        
        if return_hidden:
            return x
        if last_only:                    # generation: only the last position's logits are needed
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1), ignore_index=0)  # 0 = <pad>
        return logits, loss

In [6]:
import torch
import torch.nn as nn
import json

class SSMPromptTuningClassification(nn.Module):
    def __init__(self, base_ssm_model, d_model, num_virtual_tokens=16, num_labels=2):
        super().__init__()
        self.base_model = base_ssm_model
        self.num_virtual_tokens = num_virtual_tokens
        
        # 1. Freeze the entire base SSM
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # 2. Create learnable virtual prompt embeddings
        self.virtual_prompt = nn.Parameter(torch.randn(num_virtual_tokens, d_model))
        nn.init.normal_(self.virtual_prompt, std=0.02)
        
        # 3. Binary classification head
        self.score = nn.Linear(d_model, num_labels, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None):
        batch_size = input_ids.shape[0]
        
        # Extract base embeddings (Note: adjust 'wte' if your SSM's embedding layer is named differently)
        inputs_embeds = self.base_model.embedding(input_ids)
        
        # Prepend the learnable virtual tokens to every sequence in the batch
        prompt_embeds = self.virtual_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        full_embeds = torch.cat([prompt_embeds, inputs_embeds], dim=1)
        
        # Pass through the frozen SSM bypassing the embedding layer
        # (adjust the forward call if your SSM uses a different argument for raw embeddings)
        ssm_out = self.base_model(inputs_embeds=full_embeds, return_hidden=True) 
        
        # Handle tuple outputs if your SSM returns hidden states + cache
        if isinstance(ssm_out, tuple) or isinstance(ssm_out, list):
            ssm_out = ssm_out[0]
            
        # Pooling: Extract the last non-padded token
        # Effective sequence length shifts by num_virtual_tokens
        sequence_lengths = attention_mask.sum(dim=1) - 1 + self.num_virtual_tokens
        batch_indices = torch.arange(batch_size, device=ssm_out.device)
        pooled_representation = ssm_out[batch_indices, sequence_lengths]
        
        # Classification
        logits = self.score(pooled_representation)
        
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
            
        return {"logits": logits, "loss": loss}
        
# --- Instantiation ---
config= json.load(open("/kaggle/input/datasets/vaibhavim081203/lastbreath245/model3_ssm_output/train_config_ssm.json"))
# 1. Initialize your specific Phase 1 SSM here (e.g., base_ssm = YourSSMClass(config))
base_ssm = Model3_SSM(**config["model"])

# 2. Load the Phase 1 checkpoint into base_ssm
base_ssm.load_state_dict(torch.load("/kaggle/input/datasets/vaibhavim081203/lastbreath245/model3_ssm_output/checkpoints/best_model_ssm.pt", weights_only=True), strict=False)

# 3. Wrap it for Prompt Tuning (assuming d_model=768)
model = SSMPromptTuningClassification(base_ssm, d_model=768, num_virtual_tokens=16)

# Verify trainable parameters
print(sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable parameters")

13824 trainable parameters


In [7]:
print(model.base_model)

Model3_SSM(
  (embedding): Embedding(32000, 768)
  (layers): ModuleList(
    (0-11): 12 x MambaBlock(
      (norm): RMSNorm()
      (ssm): SSDMixer(
        (in_proj): Linear(in_features=768, out_features=3224, bias=False)
        (conv1d): Conv1d(1664, 1664, kernel_size=(4,), stride=(1,), padding=(3,), groups=1664)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=1536, out_features=768, bias=False)
      )
    )
  )
  (norm_f): RMSNorm()
  (lm_head): Linear(in_features=768, out_features=32000, bias=False)
)


## 4. Training loop and Evaluation

In [8]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW  # <-- Import from PyTorch instead
from tqdm.auto import tqdm

# --- 1. DataLoader & Padding Setup ---

# Use the safe fallback for the padding token ID defined during preprocessing
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

def ssm_collate_fn(batch):
    """Dynamically right-pads the batch to the longest sequence."""
    max_len = max(len(item['input_ids']) for item in batch)
    
    input_ids, attention_mask, labels, ids = [], [], [], []
    
    for item in batch:
        pad_len = max_len - len(item['input_ids'])
        
        # Right-padding
        input_ids.append(item['input_ids'] + [PAD_TOKEN_ID] * pad_len)
        attention_mask.append(item['attention_mask'] + [0] * pad_len)
        
        if 'label' in item:
            labels.append(item['label'])
        if 'id' in item:
            ids.append(item['id'])
            
    batch_dict = {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        # The SSM wrapper uses this mask to find the last non-padded token
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long)
    }
    if labels:
        batch_dict['labels'] = torch.tensor(labels, dtype=torch.long)
    if ids:
        batch_dict['ids'] = ids
        
    return batch_dict

# --- 2. Training Setup ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

EPOCHS = 5 # Prompt Tuning usually takes more time to align virtual epoch
LEARNING_RATE = 3e-3 # Higher LR is typical for PEFT compared to full fine-tuning

# Only pass parameters that require gradients to the optimizer
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

# --- 3. Training & Evaluation Loop ---

# --- 2. Evaluation Function ---
def evaluate_ssm(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Pass attention_mask to locate the pooling token accurately
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs['logits']
            
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Phase 4 Primary Metric
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return macro_f1


# --- 1. Reduce Batch Size ---
# Reduce this from 8 to 4 (or 2 if it still OOMs)
MICRO_BATCH_SIZE = 2 
ACCUMULATION_STEPS = 4 # Effective batch size = 2 * 4 = 8

train_loader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=MICRO_BATCH_SIZE, collate_fn=ssm_collate_fn)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=MICRO_BATCH_SIZE, collate_fn=ssm_collate_fn)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=MICRO_BATCH_SIZE, collate_fn=ssm_collate_fn)

# --- 2. Training Loop with Gradient Accumulation ---
print("Starting SSM Prompt Tuning Training with Gradient Accumulation...")
best_macro_f1 = 0.0

total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad() # Zero gradients at the START of the epoch
    
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for step, batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        # Scale the loss since we are accumulating gradients
        loss = outputs['loss'] / ACCUMULATION_STEPS
        loss.backward()
        
        # Step the optimizer only after accumulating enough gradients
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        # Track unscaled loss for logging
        total_loss += outputs['loss'].item()
        progress_bar.set_postfix({'loss': outputs['loss'].item()})
        
    avg_train_loss = total_loss / len(train_loader)
    
    # Evaluate
    val_macro_f1 = evaluate_ssm(model, val_loader)
    print(f"Epoch {epoch+1} - Avg Train Loss: {avg_train_loss:.4f} - Dev Macro-F1: {val_macro_f1:.4f}")
    
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        torch.save(model.state_dict(), "best_ssm_prompt_model.pt")
        print("--> New best model saved!")

Starting SSM Prompt Tuning Training with Gradient Accumulation...


Epoch 1/5:   0%|          | 0/18694 [00:00<?, ?it/s]

Epoch 1 - Avg Train Loss: 0.8486 - Dev Macro-F1: 0.4576
--> New best model saved!


Epoch 2/5:   0%|          | 0/18694 [00:00<?, ?it/s]

Epoch 2 - Avg Train Loss: 0.8663 - Dev Macro-F1: 0.4160


Epoch 3/5:   0%|          | 0/18694 [00:00<?, ?it/s]

Epoch 3 - Avg Train Loss: 0.7964 - Dev Macro-F1: 0.4872
--> New best model saved!


Epoch 4/5:   0%|          | 0/18694 [00:00<?, ?it/s]

Epoch 4 - Avg Train Loss: 0.7388 - Dev Macro-F1: 0.4744


Epoch 5/5:   0%|          | 0/18694 [00:00<?, ?it/s]

Epoch 5 - Avg Train Loss: 0.6818 - Dev Macro-F1: 0.4748


## 5. Prediction for Test cases and submission

In [9]:
import json
import zipfile
import torch
from tqdm.auto import tqdm

print("Generating predictions for the test set...")

# 1. Load the best weights we just trained
model.load_state_dict(torch.load("best_ssm_prompt_model.pt"))
model.eval()

predictions = []

# 2. Run inference
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs['logits']
        
        # Get binary prediction (0 or 1)
        preds = torch.argmax(logits, dim=-1).cpu().tolist()
        
        # Extract the case IDs we packed into the batch dictionary earlier
        batch_ids = batch['ids']
        
        for case_id, pred in zip(batch_ids, preds):
            predictions.append({
                "id": case_id,
                "pred": pred
            })

# 3. Save to prediction.json
output_file = "prediction.json"
with open(output_file, "w", encoding='utf-8') as f:
    json.dump(predictions, f, indent=4)

print(f"Predictions saved to {output_file}.")

# 4. Zip the file for Codabench submission
zip_filename = "submission.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # arcname ensures the file sits at the root of the zip, with no parent folders
    zipf.write(output_file, arcname=output_file)

print(f"Successfully created {zip_filename}! You are ready to submit to Codabench.")

Generating predictions for the test set...


Predicting:   0%|          | 0/759 [00:00<?, ?it/s]

Predictions saved to prediction.json.
Successfully created submission.zip! You are ready to submit to Codabench.
